# Veri üretimi · **Gemini** · Colab ve Kaggle
Gemma defterinin (`uretim.ipynb`) Gemini sürümü. **GPU gerekmez.** Önce kart, sonra hikâye üretir; dosya adına `_gemini` eklenir, Gemma dosyalarına karışmaz. Filtre (S1) ikisini birden okur.

**Anahtar:** aistudio.google.com → *Get API key* → *Create API key*.

**Colab:** Çalışma zamanı **CPU** yeterli. Sol kenardaki 🔑 **Secrets**'a `GEMINI_API_KEY` adıyla ekle (ya da *Gemini API keys* düğmesiyle içe aktar: `GOOGLE_API_KEY` adı da olur), **Notebook access** açık. 1. hücreye adını yaz → *Tümünü çalıştır*. Dosyalar Drive'a (`kaydir_veri/ham/`) gider; koparsa yeniden çalıştır, kaldığı yerden devam eder.

**Kaggle:** *File → Import Notebook* → bu dosya. **Internet: On**, GPU gerekmez. *Add-ons → Secrets* → `GEMINI_API_KEY`. Gece için **Save Version → Save & Run All**; bitince Output'tan `kaydir_veri.zip`'i indir.

Ücretsiz kotada dakikalık/günlük sınır var: dakikalık sınırda motor bekleyip yeniden dener; günlük sınır biterse defter durur, ertesi gün yeniden çalıştırınca kaldığı yerden devam eder.


## 1 · Ayar

In [ ]:
KISI = ''              # kendi adını yaz: 'mert' ya da 'sokul' (aynı ad = aynı promptlar, mutlaka farklı olsun)
GEMINI_MODEL = 'gemini-flash-latest'   # Google'ın güncel Flash modeli
KART_ADET = 1500       # bu dosyada toplam kaç kart
HIKAYE_ADET = 3000     # kaç hikâye
DAL = 'main'           # AI ve Docs depolarının dalı
SURE_SAAT = 10.5       # bu oturumda en fazla kaç saat üretilsin
KART_PAYI = 0.4        # önce kart: sürenin %40'ı karta, kalanı hikâyeye. Sadece kart için HIKAYE_ADET = 0 ve KART_PAYI = 1.0


## 2 · Hazırlık: depoları indir, klasörleri aç

In [ ]:
import os, sys, time, subprocess
assert KISI, "1. hücrede KISI'ya adını yaz ('mert' / 'sokul')"
T0 = time.time()                         # süre buradan sayılır (kurulum + model yükleme dahil)
KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))   # Colab'da da /kaggle klasörü olabiliyor; ortam değişkenine bak
ORTAM = 'kaggle' if KAGGLE else 'colab'
KOK = '/kaggle/working' if KAGGLE else '/content'
if KAGGLE:
    VERI = f'{KOK}/kaydir_veri'
    OTURUM = globals().get('OTURUM') or time.strftime('%m%d-%H%M')   # Kaggle'da her oturum yeni dosya; hücre tekrar çalışınca değişmez
else:
    from google.colab import drive; drive.mount('/content/drive')
    VERI = '/content/drive/MyDrive/kaydir_veri'
    OTURUM = time.strftime('%m%d')                                     # Colab'da her gün yeni dosya ve yeni tohum; aynı gün tekrar çalışırsa kaldığı yerden devam eder
for depo in ('AI', 'Docs'):           # depolar herkese açık; varsa güncellenir, silinmez
    yol = f'/tmp/{depo}'
    if os.path.isdir(f'{yol}/.git'):
        subprocess.run(['git', '-C', yol, 'fetch', '-q', '--depth', '1', 'origin', DAL], check=True)
        subprocess.run(['git', '-C', yol, 'checkout', '-q', '-B', DAL, 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', DAL, f'https://github.com/IleriProgramlamaProje/{depo}', yol], check=True)
for ad in [m for m in sys.modules if m == 'ai' or m.startswith('ai.')]:
    del sys.modules[ad]                # güncellenen kod yeniden yüklensin
if '/tmp/AI' not in sys.path:
    sys.path.insert(0, '/tmp/AI')      # Docs kardeş klasörde: tema.json oradan okunur
os.makedirs(f'{VERI}/ham', exist_ok=True); os.makedirs(f'{VERI}/temiz', exist_ok=True)
print(ORTAM, '·', VERI, '·', subprocess.run(['git', '-C', '/tmp/AI', 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())

## 3 · Kurulum ve Gemini anahtarı

In [ ]:
import getpass
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'google-genai'], check=True)
def gizli(ad):                          # Colab/Kaggle Secrets'tan oku; yoksa None
    try:
        if KAGGLE:
            from kaggle_secrets import UserSecretsClient; return UserSecretsClient().get_secret(ad)
        from google.colab import userdata; return userdata.get(ad)
    except Exception:
        return None
anahtar = gizli('GEMINI_API_KEY') or gizli('GOOGLE_API_KEY')
if KAGGLE and not anahtar and os.environ.get('KAGGLE_KERNEL_RUN_TYPE') == 'Batch':
    raise RuntimeError("Gece modu için Add-ons → Secrets → GEMINI_API_KEY ekle")
os.environ['GEMINI_API_KEY'] = (anahtar or getpass.getpass('Gemini API anahtarı: ')).strip()
print('Anahtar:', 'Secrets' if anahtar else 'elle girildi')


## 4 · Gemini'ye bağlan
Tek bir deneme isteği atar; anahtar ya da model adı yanlışsa burada hata verir.

In [ ]:
from ai.veri_uretimi.uretici import motor_ac, uret
MODEL = GEMINI_MODEL
motor, MOTOR_ADI = motor_ac('gemini', MODEL)
print('Deneme:', motor.uret(['Tek cümleyle merhaba de.'])[0][0][:80])
GRUP = 8                              # aynı anda 8 istek
print('Motor: gemini ·', MODEL)


## 5 · Kart üret
Kota dolarsa motor bekler; ekranda hız yazar (saatte kaç kart).


In [ ]:
BITIS = {'kart': T0 + SURE_SAAT * KART_PAYI * 3600, 'hikaye': T0 + SURE_SAAT * 3600}
HAM_K = f'{VERI}/ham/kart_{KISI}_{ORTAM}_{OTURUM}_gemini.jsonl'
HAM_H = f'{VERI}/ham/hikaye_{KISI}_{ORTAM}_{OTURUM}_gemini.jsonl'
def paketle():                      # Kaggle'da hata olsa da o ana kadar üretilen dosyalar zip'e girsin
    if KAGGLE:
        import shutil; shutil.make_archive(f'{KOK}/kaydir_veri', 'zip', VERI); print('Output → kaydir_veri.zip')
def parca_parca(tur, adet, yol):     # süre her grupta kontrol edilir; dolunca durur, kalan iş dosyada
    try:
        uret(tur, adet, yol, motor_adi=MOTOR_ADI, grup=GRUP, tohum=f'{KISI}-{ORTAM}-{OTURUM}-gemini-{tur}',
             model=MODEL, motor=motor, tolerans=True, onar=True, bitis=BITIS[tur])
    except Exception:
        paketle(); raise
parca_parca('kart', KART_ADET, HAM_K)

## 6 · Hikâye üret

In [ ]:
parca_parca('hikaye', HIKAYE_ADET, HAM_H)

## 7 · Paket
Ham dosyalar `ham/` klasöründe. Filtre (görev S1) hazır olunca `temiz/`'e süzülür; eğitimde sadece `temiz/` kullanılır.

In [ ]:
print('Kart:', HAM_K); print('Hikâye:', HAM_H)
paketle()

## Bitince
Colab: *Çalışma zamanı → Bağlantıyı kes ve çalışma zamanını sil* (birim harcamasın). Ertesi gün adetleri artır (1500 → 3000), tümünü çalıştır: kaldığı yerden devam eder.